# Sherrington-Kirkpatrick Model

The cost function of the SK model is
$$
\hat H_P = \frac{1}{\sqrt{N}}\sum_{i<j\leq N} J_{ij} \hat{Z}_i \hat{Z}_j,
$$
where the couplings $J_{ij}$ are i.i.d. standard Gaussian variables, i.e. with zero mean $\left\langle J_{ij} \right\rangle = 0$ and variance $ \left\langle J_{ij}^2 \right\rangle = J^2$.

In [ ]:
using QAOA, LinearAlgebra
import Random, Distributions
import MFAOA

using PyPlot

## QAOA

__Defining the problem by hand:__

In [ ]:
N = 4
σ2 = 1.0

Random.seed!(1)
J = rand(Distributions.Normal(0, σ2), N, N) ./ sqrt(N) 
J[diagind(J)] .= 0.0
J = UpperTriangular(J)
J = J + transpose(J)

In [ ]:
p = 2
SK_problem = QAOA.Problem(p, zeros(N), J)

__Using the wrapper function:__

In [ ]:
SK_problem = QAOA.sherrington_kirkpatrick(N, σ2, num_layers=p, seed=1)

__Gradient optimization with [Zygote](https://fluxml.ai/Zygote.jl/latest/):__

In [ ]:
learning_rate = 0.02
cost, params, probs = QAOA.optimize_parameters(SK_problem, vcat([0.5 for _ in 1:p], [0.5 for _ in 1:p]); learning_rate=learning_rate)

__Optimization with [NLopt](https://nlopt.readthedocs.io/en/latest/):__

In [ ]:
cost, params, probs = QAOA.optimize_parameters(SK_problem, vcat([0.5 for _ in 1:p], [0.5 for _ in 1:p]), :LN_COBYLA)

In [ ]:
xlabels = []
for bstr in digits.(0:2^N-1, base=2, pad=N)
    push!(xlabels, "\$|" * prod([string(b) for b in bstr]) * "\\rangle\$")
end

figure(figsize=(5, 3.2))
ax = subplot(111)
bar(0:2^N-1, probs)
ax.set_xticks(0:2^N-1)
ax.set_xticklabels(xlabels, rotation=90)
minorticks_off()
tight_layout()

## Mean-Field AOA with MFAOA.jl

The [mean-field approximate optimization algorithm](https://doi.org/10.1103/PRXQuantum.4.030335) (mean-field AOA) replaces every qubit by a classical spin vector $\mathbf{n}_i = (n_i^x, n_i^y, n_i^z)$ and runs a QAOA-like schedule on these spins. The package [MFAOA.jl](https://github.com/Quicopt/MFAOA.jl) implements it.

The SK cost function has no local fields, so flipping every spin leaves it unchanged. The mean-field AOA cannot break this symmetry by itself, so we fix the last spin to $\sigma_N = +1$. Its couplings to the other spins then act on them as local fields $h_i = J_{iN}$:

In [ ]:
h_fixed = J[1:N-1, N]        # local fields from the fixed spin σ_N = +1
J_fixed = J[1:N-1, 1:N-1]    # couplings among the remaining spins

# QAOA.jl maximizes the cost function, MFAOA.jl minimizes by default
mf_problem = MFAOA.IsingProblem(h_fixed, J_fixed; sense = :max)

__Following the spins through the schedule:__ we take MFAOA's linear schedule with $p = 1000$ steps and record every spin vector after each step.

In [ ]:
p = 1000
mf_schedule = MFAOA.linear_schedule(p, 0.5)

workspace = MFAOA.Workspace(mf_problem)
spins = workspace.state                 # spins.x[i], spins.y[i], spins.z[i] are the components of n_i
history = [[spins.x spins.y spins.z]]   # history[k][i, :] is n_i after k - 1 steps
for k in 1:p
    MFAOA.step!(workspace, mf_problem, mf_schedule.β[k], mf_schedule.γ[k])
    push!(history, [spins.x spins.y spins.z])
end

In [ ]:
# plot x, y, and z of all spins that are not fixed
times = range(0, 1, p + 1)

figure(figsize=((N - 1) * 2.2, 2))
for i in 1:N-1
    subplot(1, N - 1, i)
    for (component, label) in enumerate(["x", "y", "z"])
        plot(times, [n[i, component] for n in history], label=label)
    end
    xlim(0, 1)
    ylim(-1, 1)
    xlabel("t/T")
    ylabel("n_" * string(i))
    i == 1 && legend(fontsize=7)
end
tight_layout()

__Comparing the solutions:__ the sign of $n_i^z$ at the end of the schedule is the spin $\sigma_i$. `MFAOA.solve` runs the same schedule and returns these spins with their cost. For $N = 4$ spins, we can also find the true maximum by trying all $2^N = 16$ spin configurations:

In [ ]:
result = MFAOA.solve(mf_problem, mf_schedule)

# cost function of the spin configuration σ, maximized by both algorithms
sk_cost(σ) = sum(J[i, j] * σ[i] * σ[j] for j in 1:N for i in 1:j-1)
configurations = [[(-1)^b for b in digits(s, base=2, pad=N)] for s in 0:2^N-1]

println("QAOA, expectation value:    ", cost)
println("mean-field AOA solution:    ", result.objective)
println("maximum over all solutions: ", maximum(sk_cost, configurations))

The mean-field AOA finds the maximum. The QAOA value is an expectation over all measured bitstrings, so it stays below the maximum.

## Annealing

In [ ]:
T_anneal = 8.
p = 256
linear_schedule(t) = t / T_anneal
annealing_problem = QAOA.Problem(p, zeros(N), J);

In [ ]:
probs = anneal(annealing_problem, linear_schedule, T_anneal)